<a href="https://colab.research.google.com/github/vitoriaayres/NOTEBOOKS_COMPILADOS/blob/main/DEEP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import itertools

tf.random.set_seed(42)
np.random.seed(42)

In [4]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

num_classes = 10
y_train = tf.keras.utils.to_categorical(y_train, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])
print(f'FORMATO DE TREINO: {x_train.shape}')
print(f'FORMATO DE TESTE: {x_test.shape}')
print(f'NUMERO DE CLASSES: {num_classes}')

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2164s 13us/step
FORMATO DE TREINO: (50000, 32, 32, 3)
FORMATO DE TESTE: (10000, 32, 32, 3)
NUMERO DE CLASSES: 10


In [3]:
from tensorflow.keras import layers, models

def residual_block(x, filters, kernel_size=3, stride=1):
  'BLOCO RESIDUAL SIMPLES COM BATCH NORMALIZATION'
  shortcut = x

  x = layers.Conv2D(filters, kernel_size, padding='same', strides=stride)(x)
  x = layers.BatchNormalization()(x)
  x = layers.Activation('relu')(x)

  x = layers.Conv2D(filters, kernel_size, padding='same')(x)
  x = layers.BatchNormalization()(x)

  if shortcut.shape[-1] != filters or stride != 1:
    shortcut = layers.Conv2D(filters, 1, padding='same', strides=stride)(shortcut)
    shortcut = layers.BatchNormalization()(shortcut)

  x = layers.Add()([x, shortcut])
  x = layers.Activation('relu')(x)
  return x

inputs = layers.Input(shape=(32, 32, 3))
x = data_augmentation(inputs)
x = layers.Conv2D(64, 3, padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)

x = residual_block(x, 64)
x = residual_block(x, 64)
x = layers.MaxPooling2D(2)(x)

x = residual_block(x, 128)
x = residual_block(x, 128)
x = layers.MaxPooling2D(2)(x)

x = residual_block(x, 256)
x = residual_block(x, 256)
x = layers.MaxPooling2D(2)(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      1,792 │ sequential[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 32, 32,    │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 32, 32,    │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 32, 32,    │     36,928 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │     36,928 │ activation_3[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_4[0][0]  

 Total params: 3,050,378 (11.64 MB)

 Trainable params: 3,044,362 (11.61 MB)

 Non-trainable params: 6,016 (23.50 KB)

In [4]:
def lr_scheduler(epoch, lr):
  if epoch < 5:
    return lr
  elif epoch < 25:
    return lr * 0.95
  else:
     return lr * 0.9

lr_callback= callbacks.LearningRateScheduler(lr_scheduler, verbose= 1)

early_stopping= callbacks.EarlyStopping(
    monitor= 'val_loss',
    patience= 10,
    restore_best_weights= True
)
model_checkpoint= callbacks.ModelCheckpoint(
    'best_cifar10_model.h5',
    monitor= 'val_accuracy',
    save_best_only= True,
    mode= 'max'
)
reduce_lr= callbacks.ReduceLROnPlateau(
    monitor= 'val_loss',
    factor= 0.5,
    patience= 5,
    min_lr= 1e-7
)

model.compile(
    optimizer= optimizers.Adam(learning_rate= 0.001),
    loss= 'categorical_crossentropy',
    metrics= ['accuracy']
)

In [ ]:
history= model.fit(
    x_train, y_train,
    epochs= 5,
    batch_size= 128,
    validation_split= 0.1,
    callbacks= [early_stopping, model_checkpoint, reduce_lr, lr_callback],
    verbose= 2
)


Epoch 1: LearningRateScheduler setting learning rate to 0.0010000000474974513.
Epoch 1/5


In [ ]:
model= tf.keras.models.load_model('best_cifar10_model.h5')

test_loss, test_acc= model.evaluate(x_test, y_test, verbose= 0)
print(f'\nACURÁCIA NO TESTE: {test_acc:.4f}')
print(f'LOSS NO TESTE: {test_loss:.4f}')

x_pred= model.predict(x_test)
y_pred_classes= np.argmax(x_pred, axis=1)
y_true= np.argmax(y_test, axis=1)

print('\nRELATÓRIO DE CLASSIFICAÇÃO: ')
print(classification_report(y_true, y_pred_classes,
                            target_names= ['avião', 'automóvel', 'pássaro', 'gato',
                                           'cervo', 'cachorro', 'sapo', 'cavalo',
                                           'navio', 'caminhão']))
cm= confusion_matrix(y_true, y_pred_classes)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels= (['avião', 'automóvel', 'pássaro', 'gato',
                          'cervo', 'cachorro', 'sapo', 'cavalo',
                           'navio', 'caminhão']),
            yticklabels= ['avião', 'automóvel', 'pássaro', 'gato',
                          'cervo', 'cachorro', 'sapo', 'cavalo',
                           'navio', 'caminhão'])
plt.title('MATRIX DE CONFUSÃO - CIFAR10')
plt.ylabel('VERDADEIRO')
plt.xlabel('PREVISTO')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='TREINO')
plt.plot(history.history['val_accuracy'], label='VALIDAÇÃO')
plt.title('ACURÁCIA DURANTE O TREINAMENTO')
plt.xlabel('ÉPOCAS')
plt.ylabel('ACURÁCIA')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='TREINO')
plt.plot(history.history['val_loss'], label='VALIDAÇÃO')
plt.title('LOSS DURANTE O TREINAMENTO')
plt.xlabel('ÉPOCAS')
plt.ylabel('LOSS')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def get_activation_model(layer_name):
  return models.Model(
      inputs= model.input,
      outputs= model.get_layer(layer_name).output
  )
idx= np.random.choice(len(x_test), 5, replace=False)
sample_images= x_test[idx]
sample_labesl= y_true[idx]
sample_preds= y_pred_classes[idx]

plt.figure(figsize=(15, 3))
for i in range(5):
      plt.subplot(1, 5, i+1)
      plt.imshow(sample_images[i])
      true_label = ['avião', 'automóvel', 'pássaro', 'gato', 'cervo', 'cachorro', 'sapo', 'cavalo', 'navio', 'caminhão'][sample_labels[i]]
      pred_label = ['avião', 'automóvel', 'pássaro', 'gato', 'cervo',
                    'cachorro', 'sapo', 'cavalo', 'navio', 'caminhão'][sample_preds[i]]
      color = 'green' if sample_labels[i] == sample_preds[i] else 'red'
      plt.title(f'Verdadeiro: {true_label}\nPrevisto: {pred_label}',
                color=color, fontsize=10)
      plt.axis('off')
plt.suptitle('Exemplos de Previsões do Modelo', fontsize=16)
plt.tight_layout()
plt.show()

activation_model= get_activation_model('conv2d')
activations= activation_model.predict(np.expand_dims(sample_iamges[0], axis=0))

plt.figure(figsize=(12, 4))
for i in range(6):
  plt.subplot(2,3,i+1)
  plt.imshow(activations[0, :, :, i], cmap='viridis')
  plt.title(f'FEATURE MAP {i+1}')
  plt.axis('OFF')
plt.suptitle('ATIVAÇÕES DA PRIMEIRA CAMADA CONVOLUCIONAL', fontsize= 16)
plt.tight_layout()
plt.show()